# 14 — Versioned HTTP API

This tutorial exercises the **same** package services via FastAPI `TestClient`. It does not start a web server, access any external URL, download models, or evaluate real-world relevance. The bundled catalog is fictional.

In [ ]:
from fastapi.testclient import TestClient
from savoroute.api.app import create_app
from savoroute.data.catalog import load_catalog
from savoroute.services.discovery import DiscoveryService

app = create_app()
service = DiscoveryService(load_catalog())

## Read-only catalog and OpenAPI contract

The versioned path is `/v1`. The OpenAPI schema describes request/response types.

In [ ]:
with TestClient(app) as client:
    assert client.get("/healthz").json()["status"] == "ok"
    info = client.get("/v1/meta").json()
    assert info["catalog_synthetic"] is True
    page = client.get("/v1/items", params={"kind": "recipe", "limit": 2}).json()
    assert len(page["items"]) == 2
    schema = client.get("/openapi.json").json()
    assert "/v1/recommend" in schema["paths"]
print(info["dataset_name"], page["total"], "recipes")

## Lexical search: compare HTTP to the Python service

Ranking remains in the package, not in a notebook or HTTP handler.

In [ ]:
request = {"query": "spicy tofu noodles", "filters": {"kind": "recipe"}, "limit": 3}
with TestClient(app) as client:
    response = client.post("/v1/search", json=request)
    assert response.status_code == 200
    wire = response.json()
expected = service.search("spicy tofu noodles", limit=3)
assert wire["results"][0]["item"]["item_id"] == expected[0].item.item_id
[(hit["item"]["title"], hit["score"]) for hit in wire["results"]]

## Item-level fusion and grounded recommendations

Without an index/encoder, fusion uses BM25 only. Recommendations preserve catalog provenance and ranking evidence; no current opening-hours or dietary-safety claims are made.

In [ ]:
with TestClient(app) as client:
    fused = client.post("/v1/rank", json=request).json()
    assert fused["mode"] == "bm25_only"
    recommendations = client.post("/v1/recommend", json={**request, "preferences": {"cuisines": ["Japanese"]}}).json()
    assert recommendations["status"] == "ok"
    assert recommendations["results"][0]["retrieval_evidence"]
print(recommendations["results"][0]["explanation"])

## Validation and optional API-key access

Invalid input receives structured status codes. The example key exists only inside this test process.

In [ ]:
with TestClient(app) as client:
    invalid = client.post("/v1/search", json={"query": " "})
    assert invalid.status_code == 422
    assert invalid.json()["error"]["code"] == "validation_error"
with TestClient(create_app(api_key="tutorial-only-secret")) as client:
    assert client.get("/v1/meta").status_code == 401
    assert client.get("/v1/meta", headers={"X-API-Key": "tutorial-only-secret"}).status_code == 200
print("Validation and key checks passed")

## Next steps

Run `savoroute api serve` **locally** for live curl tests, optionally pointing it to a locally collected catalog. Do not publish the development server without TLS, operational controls, and data-license review. See `docs/http-api.md`. MCP and web UI are subsequent milestones.